# Step 1 — Load the Dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/crop_prices_SA_raw.csv", parse_dates=["date"]) # load the data and parse the date column as date type
df 

,date,year,month,week_of_year,season,crop,region,price_per_kg,rainfall_mm,temp_max_c,temp_min_c,humidity_pct,supply_index
0,2020-01-01,2020,1,1,Summer,Maize,Free State,4.26,4.6,23.0,14.1,51.7,0.330
1,2020-01-08,2020,1,2,Summer,Maize,Free State,4.25,13.2,19.5,11.4,52.7,0.406
2,2020-01-15,2020,1,3,Summer,Maize,Free State,4.31,11.1,20.2,11.6,48.3,0.444
3,2020-01-22,2020,1,4,Summer,Maize,Free State,4.27,23.8,22.2,13.7,57.6,0.523
4,2020-01-29,2020,1,5,Summer,Maize,Free State,4.41,14.0,20.9,11.2,53.4,0.296
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5003,2025-11-26,2025,11,48,Spring,Wheat,North West,7.92,0.0,16.8,7.2,42.4,0.485
5004,2025-12-03,2025,12,49,Summer,Wheat,North West,7.72,0.0,19.7,11.0,27.6,0.551
5005,2025-12-10,2025,12,50,Summer,Wheat,North West,7.45,7.5,19.0,11.0,52.1,0.490
5006,2025-12-17,2025,12,51,Summer,Wheat,North West,7.40,3.9,18.2,10.0,38.8,0.507


# Step 2 : Display info and check for missing values.

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5008 entries, 0 to 5007
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          5008 non-null   datetime64[ns]
 1   year          5008 non-null   int64         
 2   month         5008 non-null   int64         
 3   week_of_year  5008 non-null   int64         
 4   season        5008 non-null   object        
 5   crop          5008 non-null   object        
 6   region        5008 non-null   object        
 7   price_per_kg  4910 non-null   float64       
 8   rainfall_mm   4599 non-null   float64       
 9   temp_max_c    5008 non-null   float64       
 10  temp_min_c    4812 non-null   float64       
 11  humidity_pct  4762 non-null   float64       
 12  supply_index  4872 non-null   float64       
dtypes: datetime64[ns](1), float64(6), int64(3), object(3)
memory usage: 508.8+ KB


#### 2.1 check nulls

In [41]:
def display_info_and_missing_values(df):
    print("Null counts:\n", df.isnull().sum())
    print("\nNull percentage:\n", (df.isnull().sum() / len(df) * 100).round(2))


In [42]:
display_info_and_missing_values(df)

Null counts:
 date              0
year              0
month             0
week_of_year      0
season            0
crop              0
region            0
price_per_kg     98
rainfall_mm     409
temp_max_c        0
temp_min_c      196
humidity_pct    246
supply_index    136
dtype: int64

Null percentage:
 date            0.00
year            0.00
month           0.00
week_of_year    0.00
season          0.00
crop            0.00
region          0.00
price_per_kg    1.96
rainfall_mm     8.17
temp_max_c      0.00
temp_min_c      3.91
humidity_pct    4.91
supply_index    2.72
dtype: float64


#### 2.2 Handle nulls (for rainfall in millimetres, minimum temperature recorded in degrees Celsius, averege humidity percentage, market supply pressure for the crop and the price per kg)



##### For weather columns — fill with the median per region
##### (each region has its own climate so we don't mix them)

In [43]:
weather_cols = ["rainfall_mm", "temp_min_c", "humidity_pct"]
df[weather_cols] = df.groupby("region")[weather_cols].transform(
    lambda x: x.fillna(x.median())
)

display_info_and_missing_values(df[weather_cols])

Null counts:
 rainfall_mm     0
temp_min_c      0
humidity_pct    0
dtype: int64

Null percentage:
 rainfall_mm     0.0
temp_min_c      0.0
humidity_pct    0.0
dtype: float64


#####  For supply_index — fill with median per crop
##### (supply patterns differ per crop)

In [44]:

df["supply_index"] = df.groupby("crop")["supply_index"].transform(
    lambda x: x.fillna(x.median())
)

display_info_and_missing_values(df[["supply_index"]])

Null counts:
 supply_index    0
dtype: int64

Null percentage:
 supply_index    0.0
dtype: float64


#####  For price_per_kg — forward fill within each crop+region group
##### (use the last known price, most realistic for market data)

In [46]:
df["price_per_kg"] = df.groupby(["crop", "region"])["price_per_kg"].transform(
    lambda x: x.ffill().bfill()
)

display_info_and_missing_values(df[["price_per_kg"]])

Null counts:
 price_per_kg    0
dtype: int64

Null percentage:
 price_per_kg    0.0
dtype: float64


### Remaining nulls

In [47]:
print("Remaining nulls:\n", df.isnull().sum())

Remaining nulls:
 date            0
year            0
month           0
week_of_year    0
season          0
crop            0
region          0
price_per_kg    0
rainfall_mm     0
temp_max_c      0
temp_min_c      0
humidity_pct    0
supply_index    0
dtype: int64
